In [1]:
import sqlite3
import pandas as pd 
conn = sqlite3.connect(r'D:\quant_learning\week4_sql\window_practice.db')

conn.execute('DROP TABLE IF EXISTS stocks')
conn.execute('DROP TABLE IF EXISTS prices')
conn.execute('CREATE TABLE stocks (code TEXT, industry TEXT, mktcap REAL)')
conn.executemany('INSERT INTO stocks VALUES (?,?,?)', [
    ('600519','白酒',18800),('000858','白酒',5800),
    ('601398','银行',16000),('600036','银行',8800),('000001','银行',2300),
    ('300750','电池',9200),('002594','电池',7300),
    ('600276','医药',2900),('300760','医药',4500),('000538','医药',1500)])

conn.execute('CREATE TABLE prices (code TEXT, trade_date TEXT, close REAL)')
dates = ['2024-01-02','2024-01-03','2024-01-04','2024-01-05',
         '2024-01-08','2024-01-09','2024-01-10','2024-01-11']
p1 = [11.2,11.5,11.3,11.6,11.8,11.7,12.0,12.2]
p2 = [1700,1680,1720,1710,1750,1730,1760,1780]
conn.executemany('INSERT INTO prices VALUES (?,?,?)',
    [('000001',d,c) for d,c in zip(dates,p1)] +
    [('600519',d,c) for d,c in zip(dates,p2)])
conn.commit()
print(pd.read_sql('SELECT industry, COUNT(*) FROM stocks GROUP BY industry', conn))

  industry  COUNT(*)
0       医药         3
1       电池         2
2       白酒         2
3       银行         3


## Exercise

In [4]:
sql='''SELECT code, industry FROM
(SELECT code, industry, 
ROW_NUMBER() OVER (PARTITION BY industry ORDER BY mktcap DESC) AS rk
FROM stocks) s
    WHERE rk<=3
    '''
pd.read_sql(sql,conn)

,code,industry
0,300760,医药
1,600276,医药
2,000538,医药
3,300750,电池
4,002594,电池
5,600519,白酒
6,000858,白酒
7,601398,银行
8,600036,银行
9,000001,银行


In [11]:
sql='''SELECT code, trade_date, LAG(close,1) OVER(PARTITION BY code ORDER BY trade_date ASC) AS pre, 
ROUND(close-LAG(close,1) OVER(PARTITION BY code ORDER BY trade_date ASC),2) AS diff
FROM prices'''
pd.read_sql(sql,conn)

,code,trade_date,pre,diff
0,000001,2024-01-02,NaN,NaN
1,000001,2024-01-03,11.2,0.3
2,000001,2024-01-04,11.5,-0.2
3,000001,2024-01-05,11.3,0.3
4,000001,2024-01-08,11.6,0.2
5,000001,2024-01-09,11.8,-0.1
6,000001,2024-01-10,11.7,0.3
7,000001,2024-01-11,12.0,0.2
8,600519,2024-01-02,NaN,NaN
9,600519,2024-01-03,1700.0,-20.0


In [9]:
sql='''SELECT code, trade_date, 
AVG(close) OVER(PARTITION BY code ORDER BY trade_date ASC ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS ma5
FROM prices'''
pd.read_sql(sql,conn)

,code,trade_date,ma5
0,000001,2024-01-02,11.200000
1,000001,2024-01-03,11.350000
2,000001,2024-01-04,11.333333
3,000001,2024-01-05,11.400000
4,000001,2024-01-08,11.480000
5,000001,2024-01-09,11.580000
6,000001,2024-01-10,11.680000
7,000001,2024-01-11,11.860000
8,600519,2024-01-02,1700.000000
9,600519,2024-01-03,1690.000000
